# Notebook 04 — 파인튜닝 후 평가 & Before/After 비교

## 목표
QLoRA 파인튜닝된 모델을 Test Set으로 평가하고  
**Zero-shot 베이스라인 vs 파인튜닝 모델**을 정량 비교한다.

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import json, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

_cwd = Path.cwd()
ROOT = _cwd if (_cwd / "data").exists() else _cwd.parent
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS_DIR    = ROOT / "data" / "results"
MODEL_CKPT     = ROOT / "models" / "checkpoints" / "best"

DEFECT_CLASSES = [
    "crazing", "inclusion", "patches",
    "pitted_surface", "rolled-in_scale", "scratches"
]

assert (DATA_PROCESSED / "test.json").exists(), (
    "test.json 없음 — 먼저 01_dataset.ipynb를 실행하세요."
)
assert MODEL_CKPT.exists(), (
    f"체크포인트 없음: {MODEL_CKPT}\n먼저 03_finetune.ipynb를 실행해 모델을 학습시키세요."
)

with open(DATA_PROCESSED / "test.json", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Test set: {len(test_data)}개")
print(f"GPU: {torch.cuda.get_device_name(0)}")


def resolve_image(path: str) -> Path:
    p = Path(path)
    return (ROOT / p) if not p.is_absolute() else p

## 1. 파인튜닝 모델 로드

In [ ]:
from transformers import BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
processor = AutoProcessor.from_pretrained(MODEL_CKPT)

# LoRA 어댑터 로드
model = PeftModel.from_pretrained(base_model, MODEL_CKPT)
model.eval()
print("파인튜닝 모델 로드 완료")

## 2. 파인튜닝 모델 평가

In [ ]:
SYSTEM_PROMPT = (
    "당신은 금속 제품 표면 불량을 분석하는 전문 AI입니다. "
    "주어진 이미지를 분석하여 불량 유형을 정확히 판단하고 "
    "반드시 JSON 형식으로만 답변하세요."
)
INFERENCE_PROMPT = (
    "이 금속 표면 이미지를 분석하고 불량 정보를 JSON 형식으로 출력해줘.\n"
    "불량 유형은 반드시 다음 중 하나여야 해: "
    "crazing, inclusion, patches, pitted_surface, rolled-in_scale, scratches\n"
    '출력 형식: {"type": "...", "type_ko": "...", "severity": "low|medium|high", "description": "..."}'
)


def parse_output(raw: str) -> dict | None:
    raw = re.sub(r"```json\s*", "", raw)
    raw = re.sub(r"```\s*", "", raw)
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None


def infer(image_path: str) -> tuple[str, float]:
    img = Image.open(resolve_image(image_path)).convert("RGB")
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": INFERENCE_PROMPT},
        ]},
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt", padding=True).to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                                 temperature=None, top_p=None)
    elapsed = time.time() - t0
    generated = out_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(generated, skip_special_tokens=True)[0].strip(), elapsed


ft_results = []
for record in tqdm(test_data, desc="파인튜닝 모델 추론"):
    gt = json.loads(record["conversations"][1]["content"])
    raw, elapsed = infer(record["image"])
    parsed = parse_output(raw)
    pred_type = None
    pred_severity = None
    if parsed:
        pred_type = parsed.get("type", "").strip().lower()
        pred_severity = parsed.get("severity", "").strip().lower()
        if pred_type not in DEFECT_CLASSES:
            pred_type = None
    ft_results.append({
        "id": record["id"],
        "gt_type": gt["type"],
        "gt_severity": gt["severity"],
        "pred_type": pred_type,
        "pred_severity": pred_severity,
        "json_ok": parsed is not None,
        "elapsed": elapsed,
    })

ft_df = pd.DataFrame(ft_results)
ft_json_rate = ft_df["json_ok"].mean()
ft_type_acc  = (ft_df["pred_type"] == ft_df["gt_type"]).mean()
ft_sev_acc   = (ft_df["pred_severity"] == ft_df["gt_severity"]).mean()
ft_avg_time  = ft_df["elapsed"].mean()

print("=" * 45)
print("  파인튜닝 모델 결과")
print("=" * 45)
print(f"  JSON Parse Rate  : {ft_json_rate*100:.1f}%")
print(f"  Type Accuracy    : {ft_type_acc*100:.1f}%")
print(f"  Severity Accuracy: {ft_sev_acc*100:.1f}%")
print(f"  Avg Latency      : {ft_avg_time:.2f}s/image")

ft_metrics = {
    "model": "Qwen2.5-VL-7B-Instruct (QLoRA fine-tuned)",
    "json_parse_rate": round(float(ft_json_rate), 4),
    "type_accuracy": round(float(ft_type_acc), 4),
    "severity_accuracy": round(float(ft_sev_acc), 4),
    "avg_latency_sec": round(float(ft_avg_time), 3),
    "n_test": len(test_data),
}
with open(RESULTS_DIR / "finetuned_metrics.json", "w") as f:
    json.dump(ft_metrics, f, indent=2)
ft_df.to_csv(RESULTS_DIR / "finetuned_results.csv", index=False)

## 3. Before vs After 비교 시각화 (핵심 차트)

In [ ]:
with open(RESULTS_DIR / "baseline_metrics.json") as f:
    baseline = json.load(f)

metrics = ["json_parse_rate", "type_accuracy", "severity_accuracy"]
labels  = ["JSON Parse Rate", "Type Accuracy", "Severity Accuracy"]
before  = [baseline[m] * 100 for m in metrics]
after   = [ft_metrics[m] * 100 for m in metrics]
delta   = [a - b for a, b in zip(after, before)]

# 비교 바 차트
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

x = np.arange(len(labels))
w = 0.35
bars_b = axes[0].bar(x - w/2, before, w, label="Zero-shot (Before)", color="#90CAF9")
bars_a = axes[0].bar(x + w/2, after,  w, label="QLoRA Fine-tuned (After)", color="#1565C0")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=10)
axes[0].set_ylabel("(%)")
axes[0].set_title("Before vs After — 전체 지표", fontsize=13)
axes[0].set_ylim(0, 110)
axes[0].legend()
for bar in bars_b:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f"{bar.get_height():.1f}%", ha='center', fontsize=9, color="#555")
for bar in bars_a:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f"{bar.get_height():.1f}%", ha='center', fontsize=9, fontweight='bold')

# 향상폭 차트
colors = ["#4CAF50" if d > 0 else "#F44336" for d in delta]
axes[1].bar(labels, delta, color=colors)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_ylabel("개선폭 (%p)")
axes[1].set_title("파인튜닝 후 개선폭 (↑ 높을수록 좋음)", fontsize=13)
for i, (label, d) in enumerate(zip(labels, delta)):
    axes[1].text(i, d + (0.5 if d >= 0 else -1.5),
                 f"{d:+.1f}%p", ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS_DIR / "before_after_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("저장: data/results/before_after_comparison.png")

print(f"\n{'='*50}")
print(f"  최종 요약")
print(f"{'='*50}")
print(f"  Type Accuracy: {baseline['type_accuracy']*100:.1f}% → {ft_metrics['type_accuracy']*100:.1f}% ({delta[1]:+.1f}%p)")
print(f"  JSON Parse   : {baseline['json_parse_rate']*100:.1f}% → {ft_metrics['json_parse_rate']*100:.1f}% ({delta[0]:+.1f}%p)")